# 🏆 Create Gold Standard Test Set v6

**Goal:** Clean test set with REAL-WORLD distribution (not balanced)

**Strategy:**
- Remove toxic rows only
- Preserve natural ~85% zero bolus ratio

**Output:** `master_test_v6_gold.parquet`

## 1. Setup

In [ ]:
!pip install polars -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import os

print(f"Polars: {pl.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

INPUT_FILE = f"{BASE_PATH}/raw/metabonet_public_test.parquet"
OUTPUT_FILE = f"{BASE_PATH}/processed/master_test_v6_gold.parquet"

exists = "✅" if os.path.exists(INPUT_FILE) else "❌"
print(f"{exists} Input: {INPUT_FILE}")

## 2. Lazy Scanning

In [ ]:
# Columns to keep
COLS = ['CGM', 'bolus', 'carbs', 'basal', 'age', 'weight',
        'time_sin', 'time_cos', 'cgm_delta_5min', 'cgm_delta_15min',
        'is_weekend', 'gender_encoded', 'insulin_type_encoded']

print("📂 Scanning test file (lazy)...")
lf = pl.scan_parquet(INPUT_FILE)

# Get available columns
available = [c for c in COLS if c in lf.columns]
print(f"   Available columns: {len(available)}")
print(f"   Columns: {available}")

In [ ]:
# Count original rows (streaming)
print("\n📊 Counting original rows (streaming)...")
original_count = lf.select(pl.count()).collect(streaming=True).item()
print(f"   Original: {original_count:,} rows")

## 3. Toxic Row Removal

In [ ]:
def get_toxic_mask():
    """
    Toxic scenarios to DROP:
    A: CGM < 110 AND carbs == 0 AND bolus > 0
    B: CGM < 70 AND bolus > 0 (hypo with insulin)
    """
    return (
        # Scenario A: Low/Normal CGM, no food, but insulin given
        ((pl.col("CGM") < 110) & 
         (pl.col("carbs") == 0) & 
         (pl.col("bolus") > 0)) |
        
        # Scenario B: Hypoglycemia with insulin (regardless of carbs)
        ((pl.col("CGM") < 70) & 
         (pl.col("bolus") > 0))
    )

print("✅ Toxic mask defined")
print("   Scenario A: CGM<110 & carbs=0 & bolus>0")
print("   Scenario B: CGM<70 & bolus>0")

In [ ]:
# Count toxic rows before removal
print("\n🔍 Counting toxic rows...")
toxic_count = (
    lf
    .filter(get_toxic_mask())
    .select(pl.count())
    .collect(streaming=True)
    .item()
)
print(f"   Toxic rows: {toxic_count:,} ({toxic_count/original_count*100:.2f}%)")

## 4. Streaming Execution

In [ ]:
print("🧹 Cleaning and collecting (streaming)...")

df_clean = (
    lf
    .select(available)
    .filter(~get_toxic_mask())  # Keep NOT toxic
    .collect(streaming=True)
)

clean_count = len(df_clean)
print(f"\n✅ Clean rows: {clean_count:,}")
print(f"   Removed: {original_count - clean_count:,} rows")

In [ ]:
# Cast to Float32
print("📐 Casting to Float32...")
float_cols = [c for c in df_clean.columns if df_clean[c].dtype in [pl.Float64, pl.Int64]]
df_clean = df_clean.cast({c: pl.Float32 for c in float_cols})
print(f"   Casted {len(float_cols)} columns")

## 5. Export & Verification

In [ ]:
print(f"💾 Saving to: {OUTPUT_FILE}")
df_clean.write_parquet(OUTPUT_FILE)
print("✅ Saved!")

In [ ]:
# Verification
print("\n" + "=" * 60)
print("📋 VERIFICATION REPORT")
print("=" * 60)

# Zero bolus ratio
zero_count = df_clean.filter(pl.col("bolus") == 0).shape[0]
pos_count = df_clean.filter(pl.col("bolus") > 0).shape[0]

# Bolus stats
mean_bolus = df_clean["bolus"].mean()
max_bolus = df_clean["bolus"].max()

print(f"""
ROW COUNTS:
   Original:  {original_count:,}
   After Clean: {clean_count:,}
   Removed:   {original_count - clean_count:,} ({(original_count - clean_count)/original_count*100:.2f}%)

BOLUS DISTRIBUTION:
   Zero Bolus:     {zero_count:,} ({zero_count/clean_count*100:.1f}%)
   Positive Bolus: {pos_count:,} ({pos_count/clean_count*100:.1f}%)

BOLUS STATISTICS:
   Mean: {mean_bolus:.4f}
   Max:  {max_bolus:.2f}

OUTPUT FILE:
   ✅ {OUTPUT_FILE}
""")

# Check if ratio is realistic
if zero_count / clean_count > 0.80:
    print("✅ Real-world distribution preserved (~85% zero bolus)")
else:
    print("⚠️ Distribution may have shifted")

print("\n" + "=" * 60)

In [ ]:
# Final check: Any toxic rows remaining?
print("🔍 Final Toxicity Check...")

remaining_toxic = df_clean.filter(
    ((pl.col("CGM") < 70) & (pl.col("bolus") > 0))
).shape[0]

if remaining_toxic == 0:
    print("✅ No toxic rows remaining (Hypo + Bolus = 0)")
else:
    print(f"⚠️ {remaining_toxic} toxic rows still present!")